# Study 872 — Nominal-Price Illusion — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the per-book vol / skew / Sharpe (the risk-adjusted read), the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4146, 'fingerprint': '357fd262912f', 'cheap_name': 'T', 'cheap_val': 20, 'dear_name': 'CAT', 'dear_val': 1063, 'spread_bps': 2.92, 't_nw': 3.01, 't_1s': 2.83, 'lo_bps': 8.75, 'hi_bps': 5.82, 'welch_t': 1.14, 'lo_vol': 19.2, 'lo_skew': -0.26, 'lo_sharpe': 1.15, 'hi_vol': 17.8, 'hi_skew': -0.44, 'hi_sharpe': 0.82, 'placebo_obs': 2.92, 'placebo_mean': 0.019, 'placebo_sd': 1.218, 'placebo_p': 0.014, 'placebo_sigma': 2.4, 'placebo_draws': 1000, 'era_early_bps': 2.66, 'era_early_t': 2.11, 'era_early_n': 2012, 'era_late_bps': 3.18, 'era_late_t': 2.16, 'era_late_n': 2134, 'timer_1_gross': 2.92, 'timer_1_cost': 2.14, 'timer_1_net': 0.79, 'timer_1_t': 0.76, 'timer_5_gross': 2.92, 'timer_5_cost': 10.14, 'timer_5_net': -7.21, 'timer_5_t': -6.98, 'null_mean_t': -0.15, 'null_sd_t': 0.92, 'null_fire': 0, 'planted_t': -3.69, 'planted_welch': -4.01, 'planted_lo_vol': 19.8, 'planted_lo_skew': 0.89, 'planted_hi_vol': 6.3, 'planted_hi_skew': 0.33}

## The headline — long-cheap / short-dear spread (`lo − hi`)

Daily equal-weight cheapest-30% minus priciest-30% price-level spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : cheap {R['lo_bps']:+.2f} vs dear {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")

spread        : +2.92 bps/day  NW(10) t = +3.01  one-sample t = +2.83
books         : cheap +8.75 vs dear +5.82 bps (Welch t = +1.14)


## The risk-adjusted lottery read — more risk for less reward?

The over-priced-lottery hypothesis is about *risk-adjusted* underperformance. It fails on both legs: the cheap book is only slightly more volatile, and its Sharpe is *higher*, not lower.

In [3]:
print(f"cheap book : vol {R['lo_vol']:.1f}%/yr  skew {R['lo_skew']:+.2f}  Sharpe {R['lo_sharpe']:+.2f}")
print(f"dear  book : vol {R['hi_vol']:.1f}%/yr  skew {R['hi_skew']:+.2f}  Sharpe {R['hi_sharpe']:+.2f}")

cheap book : vol 19.2%/yr  skew -0.26  Sharpe +1.15
dear  book : vol 17.8%/yr  skew -0.44  Sharpe +0.82


## Placebo — column-permute the forward returns (1,000 permutations, two-sided)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> two-sided p = {R['placebo_p']:.4f} "
      f"(~{R['placebo_sigma']:.1f} sigma into the right tail)")

observed +2.92 bps vs placebo mean +0.019 (sd 1.218) -> two-sided p = 0.0140 (~2.4 sigma into the right tail)


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print('the (wrong-sign) out-performance of cheap names holds in BOTH halves -> robust None, not noise')

2010-2017 (n=2012): +2.66 bps  NW t = +2.11
2018-2026 (n=2134): +3.18 bps  NW t = +2.16
the (wrong-sign) out-performance of cheap names holds in BOTH halves -> robust None, not noise


## The timer — can you get paid for it (even the reversed book)?

2 sides × one-way cost × NAV per day on the long-short book; short (dear) pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")
print('even the data-mined sign-flip is insignificant at 1 bp and negative at 5 bps -> Mirage')

 1 bp one-way: gross +2.92 -> net +0.79 bps/day (cost 2.14/day, t=+0.76)
5 bps one-way: gross +2.92 -> net -7.21 bps/day (cost 10.14/day, t=-6.98)
even the data-mined sign-flip is insignificant at 1 bp and negative at 5 bps -> Mirage


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted under-earn relation (a *negative* spread), while planting the lottery look.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from nominal_price import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=872+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=872, n_assets=40, n_days=1500))
print(f"planted (edge=0.0016): NW t = {planted['t_nw']:+.2f} (negative = cheap under-earns), Welch t = {planted['welch_t']:+.2f}")
print(f"planted lottery look: cheap vol {planted['lo_vol']*100:.1f}% skew {planted['lo_skew']:+.2f}  vs  dear vol {planted['hi_vol']*100:.1f}% skew {planted['hi_skew']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.40 (sd 0.87), |t|>=2 in 0/8


planted (edge=0.0016): NW t = -3.69 (negative = cheap under-earns), Welch t = -4.01
planted lottery look: cheap vol 19.8% skew +0.89  vs  dear vol 6.3% skew +0.33


## Verdict

- **Signal — None.** The nominal-price money-illusion premium does **not** replicate on 50 liquid US mega-caps: the long-cheap / short-dear spread is **+2.92 bps/day** (NW *t* = **+3.01**) — significant but *opposite in sign* to the claim (cheap names out-earned, with a *higher* Sharpe, +1.15 vs +0.82), holding in both eras (*t* = +2.11 / +2.16), ≈2.4σ into the right tail of a 1,000-permutation placebo. The synthetic control recovers a *planted* under-earn relation cleanly (*t* = -3.69, fires on 0/20 nulls), so the sign-reversal is real, not machinery. Mega-caps are *rarely cheap* — the lottery segment is absent (honest low power).
- **Tradability — Mirage.** Even the data-mined sign-flip dies: net **+0.79 bps/day** at 1 bp one-way but *insignificant* (*t* = +0.76), and **-7.21 bps/day** at 5 bps. No version survives the 2.14 bps/day round-trip friction.